# 一、构建卷积神经网络参数

- 卷积神经网络中的输入和层与传统神经网络有些区别， 需重新设计， 训练模块基本一致


In [9]:
# 基本环境

import torch;
import torch.nn as nn;
import torch.optim as optim;
import torch.nn.functional as F;
#  datasets  处理数据集
#  transforms 预处理操作的
from torchvision import datasets, transforms;
import matplotlib.pyplot as plt;
import numpy as np;

%matplotlib inline



# 二、首先读取数据集

- 分别构建训练集和测试集（验证集）
- DataLoader来迭代取数据 

In [8]:
# 定义超参数

#  1.  图像的总尺寸 28 * 28
input_size = 28; 
#  2. 标签的种类数
num_classses = 10;

# 3. 训练的总循环周期
num_epochs = 3;
# 4. 一个最（批次）的大小 64张图片
batch_size = 64;

# 训练集 
train_dataset = datasets.MNIST('data/', 
                               download=True, 
                               train=True, 
                               transform=transforms.ToTensor());

# 测试集
test_dataset = datasets.MNIST('data/', 
                               download=True, 
                               train=False, 
                               transform=transforms.ToTensor());

# 构建batch数据

train_loader  = torch.utils.data.DataLoader(dataset = train_dataset,
                                           batch_size = batch_size,
                                           shuffle=True);

test_loader = torch.utils.data.DataLoader(dataset = test_dataset,
                                           batch_size = batch_size,
                                           shuffle=True);



Failed to download (trying next):
HTTP Error 404: Not Found



100.0%


Extracting data/MNIST\raw\train-images-idx3-ubyte.gz to data/MNIST\raw

Failed to download (trying next):
HTTP Error 404: Not Found



100.0%


Extracting data/MNIST\raw\train-labels-idx1-ubyte.gz to data/MNIST\raw

Failed to download (trying next):
HTTP Error 404: Not Found



100.0%


Extracting data/MNIST\raw\t10k-images-idx3-ubyte.gz to data/MNIST\raw

Failed to download (trying next):
HTTP Error 404: Not Found



100.0%

Extracting data/MNIST\raw\t10k-labels-idx1-ubyte.gz to data/MNIST\raw



# 三、卷积网络模块构建

1.  一般卷积层， relu层， 池化层可以写成一个套餐
2.  注意卷积最后结果还是一个特征图， 需要把图转换成向量才能做分类或者回归任务

In [12]:
class  CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__();
        self.conv1 = nn.Sequential(          # 输入大小 （1， 28， 28）
            # 2D卷积9
            nn.Conv2d(
                in_channels = 1,             # 灰度图
                out_channels = 16,           # 要得到几多少个特征图
                kernel_size = 5,             # 卷积核大小
                stride = 1,                  # 步长
                padding = 2,                 # 如果希望卷积后大小更原来一样 需要设置padding=(kernel_size-1)/2 if stride = 1
            ),                               # 输出的特征图为（16， 28， 28）
            nn.ReLU(),                       # relu层
            nn.MaxPool2d(kernel_size = 2),   # 进行池化操作（2*2区域） 输出结果为（16， 14， 14);
            
        );

        self.conv2 = nn.Sequential(           # 下一个套餐的输入(16, 14, 14)
            nn.Conv2d(16, 32, 5, 1, 2),       # 输出（32， 14， 14）
            nn.ReLU(),                        # relu层
            nn.Conv2d(32, 32, 5, 1, 2),       # 
            nn.ReLU(),
            nn.MaxPool2d(2),                  # 输出（32， 7， 7）
        );

        self.conv3 = nn.Sequential(           # 下一个套餐的输入(16, 14, 14)
            nn.Conv2d(32, 64, 5, 1, 2),       # 输出（32， 14， 14）
            nn.ReLU(),                        # 输出 （32， 7， 7）
        );


        self.out = nn.Linear(64 * 7 * 7, 10);    # 全连接层得到的结果


    # 前向传播
    def forward(self, x):
        x = self.conv1(x);
        x = self.conv2(x);
        x = self.conv3(x);
        x = x.view(x.size(0), -1);             # flatten 操作 结果为：（batch_size， 32 * 7 * 7）
        output = self.out(x); 
        return output;

# 四、准确率作为评估标准


In [14]:
def accuracy(predictions, labels):
    pred = torch.max(predictions.data, 1)[1];
    # 预测值与标签是否相等的
    rights = pred.eq(labels.data.view_as(pred)).sum();
    return rights, len(labels);

# 五、训练网络模型


In [ ]:
# 实例化

net = CNN();

# 损失函数
criterion = nn.CrossEntropyLoss();

# 优化器    定义优化器 普通的随机梯度下降算法   lr : 学习率
optimizer  = optim.Adam(net.parameters(), lr=0.001);

# 开始训练循环
for epoch in range(num_epochs):
    # 当前epoch的结果保存下来
    train_rights = [];

    #针对容器中的每一个批进行循环
    for batch_idx, (data, target) in enumerate(train_loader): 
        net.train();
        output = net(data);
        loss = criterion(output, target);
        optimizer.zero_grad();
        loss.backward();
        optimizer.step();
        right = accuracy(output, target);
        train_rights.append(right);

        if batch_idx % 100 == 0:
            net.eval();
            val_rights = [];

            for (data, target) in test_loader:
                output = net(data);
                right = accuracy(output, target);
                val_rights.append(right);

            # 准确率计算
            train_r = (sum([tup[0] for tup in train_rights]), sum([tup[1] for tup in train_rights]));
            val_r =  (sum([tup[0] for tup in val_rights]), sum([tup[1] for tup in val_rights]));

            print('当前epoch：{} [{}/{} ({:.0f}%)]\t 损失: {:.6f}\t 训练集准确率:{:.2f}%\t 测试集准确率:{:.2f}%'.format(
                epoch, batch_idx * batch_size, len(train_loader.dataset),
                100. * batch_idx / len(train_loader),
                loss.data,
                100. *train_r[0].numpy() / train_r[1],
                100. * val_r[0].numpy() / val_r[1]
            ));

当前epoch：0 [0/60000 (0%)]	 损失: 2.304943	 训练集准确率:9.38%	 测试集准确率:9.75%
当前epoch：0 [6400/60000 (11%)]	 损失: 0.237522	 训练集准确率:77.78%	 测试集准确率:94.02%
当前epoch：0 [12800/60000 (21%)]	 损失: 0.076119	 训练集准确率:86.25%	 测试集准确率:96.01%
当前epoch：0 [19200/60000 (32%)]	 损失: 0.150745	 训练集准确率:89.65%	 测试集准确率:97.10%
当前epoch：0 [25600/60000 (43%)]	 损失: 0.069411	 训练集准确率:91.51%	 测试集准确率:98.07%
当前epoch：0 [32000/60000 (53%)]	 损失: 0.017874	 训练集准确率:92.71%	 测试集准确率:98.01%
当前epoch：0 [38400/60000 (64%)]	 损失: 0.062962	 训练集准确率:93.54%	 测试集准确率:98.29%
当前epoch：0 [44800/60000 (75%)]	 损失: 0.179680	 训练集准确率:94.11%	 测试集准确率:98.54%
当前epoch：0 [51200/60000 (85%)]	 损失: 0.063759	 训练集准确率:94.59%	 测试集准确率:98.31%
当前epoch：0 [57600/60000 (96%)]	 损失: 0.136003	 训练集准确率:94.96%	 测试集准确率:98.41%
当前epoch：1 [0/60000 (0%)]	 损失: 0.010682	 训练集准确率:100.00%	 测试集准确率:98.47%
当前epoch：1 [6400/60000 (11%)]	 损失: 0.081811	 训练集准确率:98.73%	 测试集准确率:98.68%
当前epoch：1 [12800/60000 (21%)]	 损失: 0.004789	 训练集准确率:98.68%	 测试集准确率:98.38%
当前epoch：1 [19200/60000 (32%)]	 损失: 0.007084	 训练集准确率

# 提高

1. 再加入一层卷积， 效果怎么样？
2. 当前任务重为什么全连接层3277 其中每一个数字代表什么含义